# WNet3D Architecture
ALL BASED ON https://doi.org/10.7554/eLife.99848.4

Fully parameterizable WNet3D model extracted from CellSeg3D.

**WNet3D** cascades two identical U-Net halves:
- **Encoder U-Net** (`U_enc`): input image → K-class segmentation (with softmax)
- **Decoder U-Net** (`U_dec`): segmentation map → reconstructed image (no softmax)

Each U-Net uses a 4-level encoder-decoder with channels `[64, 128, 256, 512]`.

All layers (activation, normalization, dropout, kernel_size, padding_mode) are parameterizable.

---
## 1. Imports & Defaults

In [2]:
import torch
import torch.nn as nn
from typing import List, Optional

# ─── Defaults ─────────────────────────────────────────────────────────

DEFAULT_NUM_GROUPS = 4          # GroupNorm groups used throughout the network
DEFAULT_DROPOUT = 0.65          # CellSeg3D default dropout rate
DEFAULT_ACTIVATION = "relu"
DEFAULT_NORM = "group"
DEFAULT_DROPOUT_TYPE = "standard"
DEFAULT_KERNEL_SIZE = 3         # spatial conv kernel size (CellSeg3D uses 3)
DEFAULT_PADDING_MODE = "zeros"  # padding mode for spatial convs
                                # options: 'zeros', 'reflect', 'replicate', 'circular'

print("Defaults loaded.")
print(f"  kernel_size={DEFAULT_KERNEL_SIZE}, padding_mode='{DEFAULT_PADDING_MODE}'")
print(f"  activation='{DEFAULT_ACTIVATION}', norm='{DEFAULT_NORM}'")
print(f"  dropout={DEFAULT_DROPOUT}, dropout_type='{DEFAULT_DROPOUT_TYPE}'")
print(f"  num_groups={DEFAULT_NUM_GROUPS}")

Defaults loaded.
  kernel_size=3, padding_mode='zeros'
  activation='relu', norm='group'
  dropout=0.65, dropout_type='standard'
  num_groups=4


---
## 2. Factory Functions

These let you swap activation / normalization / dropout by changing a string.

In [3]:
def get_activation(activation: str = DEFAULT_ACTIVATION) -> nn.Module:
    """Return an activation layer by name.

    Supported: 'relu', 'leaky_relu', 'elu', 'gelu'.
    """
    if activation == "relu":
        return nn.ReLU(inplace=True)
    elif activation == "leaky_relu":
        return nn.LeakyReLU(inplace=True)
    elif activation == "elu":
        return nn.ELU(inplace=True)
    elif activation == "gelu":
        return nn.GELU()
    else:
        raise ValueError(f"Unknown activation: {activation}")


def get_norm_layer(norm_type: str,
                   num_channels: int,
                   num_groups: int = DEFAULT_NUM_GROUPS) -> nn.Module:
    """Return a normalization layer by name.

    Supported: 'group', 'batch', 'instance'.

    Args:
        norm_type:    which normalization to use.
        num_channels: number of feature channels to normalize.
        num_groups:   groups for GroupNorm (ignored by batch/instance).
    """
    if norm_type == "group":
        return nn.GroupNorm(num_groups=num_groups, num_channels=num_channels)
    elif norm_type == "batch":
        return nn.BatchNorm3d(num_channels)
    elif norm_type == "instance":
        return nn.InstanceNorm3d(num_channels)
    else:
        raise ValueError(
            f"Unknown norm_type: {norm_type}. Choose 'group', 'batch', or 'instance'."
        )


def get_dropout_layer(p: float = DEFAULT_DROPOUT,
                      dropout_type: str = DEFAULT_DROPOUT_TYPE) -> nn.Module:
    """Return a dropout layer by name.

    Supported:
      'standard' - nn.Dropout3d: drops entire 3D feature channels.
      'spatial'  - nn.Dropout2d: drops entire 2D feature maps (channel-wise).

    Args:
        p:            dropout probability.
        dropout_type: 'standard' or 'spatial'.
    """
    if dropout_type == "standard":
        return nn.Dropout3d(p=p)
    elif dropout_type == "spatial":
        return nn.Dropout2d(p=p)
    else:
        raise ValueError(
            f"Unknown dropout_type: {dropout_type}. Choose 'standard' or 'spatial'."
        )


print("Factory functions defined: get_activation, get_norm_layer, get_dropout_layer")

Factory functions defined: get_activation, get_norm_layer, get_dropout_layer


---
## 3. Building Blocks

Three block types make up the U-Net:

| Block | Where used | Conv pattern |
|-------|------------|--------------|
| **InBlock** | Top of encoder | `spatial_conv -> spatial_conv` |
| **DownBlock** | All other encoder/decoder levels | `spatial_conv -> 1x1x1` (x2) |
| **OutBlock** | Bottom of decoder (final output) | `spatial_conv -> spatial_conv -> 1x1x1` |

In [4]:
class InBlock(nn.Module):
    """First block at the top of the U-Net encoder.

    Two spatial convolutions, each followed by activation -> dropout -> norm.
    Transforms: in_channels -> out_channels -> out_channels.

    Args:
        kernel_size:  spatial conv kernel size (default 3).
        padding_mode: padding fill strategy ('zeros', 'reflect', 'replicate', 'circular').
    """

    def __init__(self, in_ch: int, out_ch: int,
                 kernel_size: int = DEFAULT_KERNEL_SIZE,
                 padding_mode: str = DEFAULT_PADDING_MODE,
                 activation: str = DEFAULT_ACTIVATION,
                 norm_type: str = DEFAULT_NORM,
                 num_groups: int = DEFAULT_NUM_GROUPS,
                 dropout: float = DEFAULT_DROPOUT,
                 dropout_type: str = DEFAULT_DROPOUT_TYPE):
        super().__init__()
        padding = kernel_size // 2  # keeps spatial size unchanged
        self.block = nn.Sequential(
            # Conv 1: expand input channels to out_ch
            nn.Conv3d(in_ch, out_ch, kernel_size=kernel_size,
                      padding=padding, padding_mode=padding_mode),
            get_activation(activation),
            get_dropout_layer(dropout, dropout_type),
            get_norm_layer(norm_type, out_ch, num_groups),
            # Conv 2: refine features at out_ch
            nn.Conv3d(out_ch, out_ch, kernel_size=kernel_size,
                      padding=padding, padding_mode=padding_mode),
            get_activation(activation),
            get_dropout_layer(dropout, dropout_type),
            get_norm_layer(norm_type, out_ch, num_groups),
        )

    def forward(self, x):
        return self.block(x)


print("InBlock defined.")

InBlock defined.


In [5]:
class DownBlock(nn.Module):
    """Encoder/decoder block used at every level except input and output.

    Two pairs of (spatial conv -> 1x1x1 channel projection),
    each followed by activation -> dropout -> norm.
    Transforms: in_channels -> out_channels.

    Args:
        kernel_size:  spatial conv kernel size (default 3). The 1x1x1 projection
                      always uses kernel_size=1 regardless.
        padding_mode: padding fill strategy for spatial convs.
    """

    def __init__(self, in_ch: int, out_ch: int,
                 kernel_size: int = DEFAULT_KERNEL_SIZE,
                 padding_mode: str = DEFAULT_PADDING_MODE,
                 activation: str = DEFAULT_ACTIVATION,
                 norm_type: str = DEFAULT_NORM,
                 num_groups: int = DEFAULT_NUM_GROUPS,
                 dropout: float = DEFAULT_DROPOUT,
                 dropout_type: str = DEFAULT_DROPOUT_TYPE):
        super().__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            # Pair 1: spatial conv + channel projection
            nn.Conv3d(in_ch, in_ch, kernel_size=kernel_size,
                      padding=padding, padding_mode=padding_mode),
            nn.Conv3d(in_ch, out_ch, kernel_size=1),             # 1x1x1 always
            get_activation(activation),
            get_dropout_layer(dropout, dropout_type),
            get_norm_layer(norm_type, out_ch, num_groups),
            # Pair 2: spatial conv + channel projection
            nn.Conv3d(out_ch, out_ch, kernel_size=kernel_size,
                      padding=padding, padding_mode=padding_mode),
            nn.Conv3d(out_ch, out_ch, kernel_size=1),            # 1x1x1 always
            get_activation(activation),
            get_dropout_layer(dropout, dropout_type),
            get_norm_layer(norm_type, out_ch, num_groups),
        )

    def forward(self, x):
        return self.block(x)


print("DownBlock defined.")

DownBlock defined.


In [6]:
class OutBlock(nn.Module):
    """Final block at the top of the U-Net decoder.

    Two spatial convolutions (intermediate channels configurable, default 64),
    each followed by activation -> dropout -> norm, then a 1x1x1
    projection to the output channels.

    Args:
        kernel_size:  spatial conv kernel size (default 3). The final 1x1x1
                      projection always uses kernel_size=1.
        padding_mode: padding fill strategy for spatial convs.
        mid_channels: intermediate channel count (default 64, matches CellSeg3D).
    """

    def __init__(self, in_ch: int, out_ch: int,
                 kernel_size: int = DEFAULT_KERNEL_SIZE,
                 padding_mode: str = DEFAULT_PADDING_MODE,
                 activation: str = DEFAULT_ACTIVATION,
                 norm_type: str = DEFAULT_NORM,
                 num_groups: int = DEFAULT_NUM_GROUPS,
                 dropout: float = DEFAULT_DROPOUT,
                 dropout_type: str = DEFAULT_DROPOUT_TYPE,
                 mid_channels: int = 64):
        super().__init__()
        mid = mid_channels  # intermediate channels before final projection
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, mid, kernel_size=kernel_size,
                      padding=padding, padding_mode=padding_mode),
            get_activation(activation),
            get_dropout_layer(dropout, dropout_type),
            get_norm_layer(norm_type, mid, num_groups),
            nn.Conv3d(mid, mid, kernel_size=kernel_size,
                      padding=padding, padding_mode=padding_mode),
            get_activation(activation),
            get_dropout_layer(dropout, dropout_type),
            get_norm_layer(norm_type, mid, num_groups),
            nn.Conv3d(mid, out_ch, kernel_size=1),  # 1x1x1 final projection
        )

    def forward(self, x):
        return self.block(x)


print("OutBlock defined.")

OutBlock defined.


---
## 4. UNet3DHalf

Single U-Net half of the WNet3D.

```
Architecture (channels=[64, 128, 256, 512]):

  Encoder:                          Decoder:
    in --> InBlock --> 64  ----skip----> cat + OutBlock --> out
           |                                  ^
           v  MaxPool                  ConvT  |
    DownBlock --> 128  ------skip----> cat + DownBlock --> 128
           |                                  ^
           v  MaxPool                  ConvT  |
    DownBlock --> 256  ------skip----> cat + DownBlock --> 256
           |                                  ^
           v  MaxPool                  ConvT  |
    DownBlock --> 512 (bottleneck) ----+
```

In [7]:
class UNet3DHalf(nn.Module):
    """Single U-Net half of the WNet3D.

    Args:
        in_channels:      number of input channels.
        out_channels:     number of output channels.
        use_softmax:      apply Softmax(dim=1) on output (True for encoder U-Net).
        kernel_size:      spatial conv kernel size for all blocks (default 3).
        padding_mode:     padding fill strategy for all spatial convs.
        activation:       activation function name.
        norm_type:        normalization layer name.
        num_groups:       groups for GroupNorm.
        dropout:          dropout probability.
        dropout_type:     dropout variant.
        out_mid_channels: intermediate channels in OutBlock (default 64).
    """

    def __init__(self, in_channels: int, out_channels: int,
                 use_softmax: bool = False,
                 kernel_size: int = DEFAULT_KERNEL_SIZE,
                 padding_mode: str = DEFAULT_PADDING_MODE,
                 activation: str = DEFAULT_ACTIVATION,
                 norm_type: str = DEFAULT_NORM,
                 num_groups: int = DEFAULT_NUM_GROUPS,
                 dropout: float = DEFAULT_DROPOUT,
                 dropout_type: str = DEFAULT_DROPOUT_TYPE,
                 out_mid_channels: int = 64):
        super().__init__()
        # Channel progression: [64, 128, 256, 512]
        channels = [64, 128, 256, 512]
        self.use_softmax = use_softmax

        # Common kwargs passed to every block
        bkw = dict(kernel_size=kernel_size, padding_mode=padding_mode,
                    activation=activation, norm_type=norm_type,
                    num_groups=num_groups, dropout=dropout,
                    dropout_type=dropout_type)

        # --- Encoder ---
        self.in_block = InBlock(in_channels, channels[0], **bkw)
        self.pool = nn.MaxPool3d(kernel_size=2)

        # 3 encoder stages: 64->128, 128->256, 256->512 (bottleneck)
        self.enc1 = DownBlock(channels[0], channels[1], **bkw)
        self.enc2 = DownBlock(channels[1], channels[2], **bkw)
        self.enc3 = DownBlock(channels[2], channels[3], **bkw)  # bottleneck

        # --- Decoder ---
        # Transposed convolutions for 2x upsampling
        self.up1 = nn.ConvTranspose3d(channels[3], channels[2], kernel_size=2, stride=2)
        self.up2 = nn.ConvTranspose3d(channels[2], channels[1], kernel_size=2, stride=2)
        self.up3 = nn.ConvTranspose3d(channels[1], channels[0], kernel_size=2, stride=2)

        # Decoder blocks (input = upsampled + skip concatenation -> double channels)
        self.dec1 = DownBlock(channels[3], channels[2], **bkw)      # 512->256
        self.dec2 = DownBlock(channels[2], channels[1], **bkw)      # 256->128
        self.out_block = OutBlock(channels[1], out_channels,
                                  mid_channels=out_mid_channels, **bkw)  # 128->out

        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        # Encoder -- collect skip connections
        s0 = self.in_block(x)                  # [B, 64,  D,   H,   W  ]
        s1 = self.enc1(self.pool(s0))          # [B, 128, D/2, H/2, W/2]
        s2 = self.enc2(self.pool(s1))          # [B, 256, D/4, H/4, W/4]
        bot = self.enc3(self.pool(s2))         # [B, 512, D/8, H/8, W/8]

        # Decoder -- upsample, concat skip, apply block
        x = torch.cat([s2, self.up1(bot)], dim=1)  # [B, 512, ...]
        x = self.dec1(x)                            # [B, 256, ...]

        x = torch.cat([s1, self.up2(x)], dim=1)    # [B, 256, ...]
        x = self.dec2(x)                            # [B, 128, ...]

        x = torch.cat([s0, self.up3(x)], dim=1)    # [B, 128, ...]
        x = self.out_block(x)                       # [B, out, ...]

        if self.use_softmax:
            x = self.softmax(x)

        return x


print("UNet3DHalf defined.")

UNet3DHalf defined.


---
## 5. WNet3D

Two cascaded U-Nets for unsupervised 3D segmentation.

```
input image --> encoder U-Net --> K-class probabilities (softmax)
                                        |
                                        v
                  decoder U-Net --> reconstructed image
```

In [8]:
class WNet3D(nn.Module):
    """WNet3D: two cascaded U-Nets for unsupervised 3D segmentation.

    All internal layers (activation, norm, dropout, kernel_size, padding_mode)
    are parameterizable.

    Args:
        in_channels:      input image channels (1 for grayscale).
        out_channels:     reconstruction output channels (1 for grayscale).
        num_classes:      segmentation classes K (default 2).
        kernel_size:      spatial conv kernel size for all blocks (default 3).
        padding_mode:     padding fill strategy ('zeros', 'reflect', 'replicate', 'circular').
        activation:       activation function name ('relu', 'leaky_relu', 'elu', 'gelu').
        norm_type:        normalization type ('group', 'batch', 'instance').
        num_groups:       groups for GroupNorm (default 4).
        dropout:          dropout probability (default 0.65).
        dropout_type:     dropout variant ('standard', 'spatial').
        out_mid_channels: intermediate channels in the OutBlock (default 64).
    """

    def __init__(self, in_channels: int = 1, out_channels: int = 1,
                 num_classes: int = 2,
                 kernel_size: int = DEFAULT_KERNEL_SIZE,
                 padding_mode: str = DEFAULT_PADDING_MODE,
                 activation: str = DEFAULT_ACTIVATION,
                 norm_type: str = DEFAULT_NORM,
                 num_groups: int = DEFAULT_NUM_GROUPS,
                 dropout: float = DEFAULT_DROPOUT,
                 dropout_type: str = DEFAULT_DROPOUT_TYPE,
                 out_mid_channels: int = 64):
        super().__init__()

        common = dict(kernel_size=kernel_size, padding_mode=padding_mode,
                      activation=activation, norm_type=norm_type,
                      num_groups=num_groups, dropout=dropout,
                      dropout_type=dropout_type,
                      out_mid_channels=out_mid_channels)

        # Encoder U-Net: image -> K-class soft segmentation
        self.encoder = UNet3DHalf(
            in_channels=in_channels,
            out_channels=num_classes,
            use_softmax=True,
            **common,
        )

        # Decoder U-Net: K-class segmentation -> reconstructed image
        self.decoder = UNet3DHalf(
            in_channels=num_classes,
            out_channels=out_channels,
            use_softmax=False,
            **common,
        )

    def forward(self, x):
        """Returns (segmentation, reconstruction)."""
        seg = self.encoder(x)
        rec = self.decoder(seg)
        return seg, rec

    def forward_encoder(self, x):
        """Encoder only -- returns K-class probability map."""
        return self.encoder(x)

    def forward_decoder(self, seg):
        """Decoder only -- reconstructs image from segmentation."""
        return self.decoder(seg)


print("WNet3D defined.")

WNet3D defined.


---
## 6. Smoke Test

Instantiate the model, pass a random tensor, verify shapes and parameter counts.

In [9]:
def smoke_test(model_kwargs=None, input_size=(1, 1, 32, 32, 32)):
    """Quick test: build model, forward pass, check shapes."""
    if model_kwargs is None:
        model_kwargs = {}  # use all defaults

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = WNet3D(**model_kwargs).to(device)

    # Parameter count
    total = sum(p.numel() for p in model.parameters())
    enc_p = sum(p.numel() for p in model.encoder.parameters())
    dec_p = sum(p.numel() for p in model.decoder.parameters())
    print(f"Parameters: {total:,} total  (encoder: {enc_p:,}, decoder: {dec_p:,})")

    # Forward pass
    x = torch.randn(*input_size, device=device)
    model.eval()
    with torch.no_grad():
        seg, rec = model(x)

    print(f"Input:          {list(x.shape)}")
    print(f"Segmentation:   {list(seg.shape)}  (softmax sum={seg.sum(dim=1).mean():.4f})")
    print(f"Reconstruction: {list(rec.shape)}")
    print("OK")

    del model, x, seg, rec
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# --- Default config (matches CellSeg3D) ---
print("=== Default (CellSeg3D) ===")
smoke_test()

print()

# --- Custom config ---
print("=== Custom: kernel=5, reflect pad, LeakyReLU, BatchNorm, dropout=0.3 ===")
smoke_test(dict(
    kernel_size=5,
    padding_mode="reflect",
    activation="leaky_relu",
    norm_type="batch",
    dropout=0.3,
    out_mid_channels=32,
))

=== Default (CellSeg3D) ===
Parameters: 50,531,843 total  (encoder: 25,265,090, decoder: 25,266,753)
Input:          [1, 1, 32, 32, 32]
Segmentation:   [1, 2, 32, 32, 32]  (softmax sum=1.0000)
Reconstruction: [1, 1, 32, 32, 32]
OK

=== Custom: kernel=5, reflect pad, LeakyReLU, BatchNorm, dropout=0.3 ===
Parameters: 216,546,723 total  (encoder: 108,269,378, decoder: 108,277,345)
Input:          [1, 1, 32, 32, 32]
Segmentation:   [1, 2, 32, 32, 32]  (softmax sum=1.0000)
Reconstruction: [1, 1, 32, 32, 32]
OK


---
## 7. Loss Functions (CellSeg3D)

WNet3D training uses **two losses** combined as a weighted sum:

$$\mathcal{L} = \alpha \cdot \mathcal{L}_{\text{NCuts}} + \beta \cdot \mathcal{L}_{\text{Rec}}$$

| Loss | Applied to | Purpose | Default weight |
|------|-----------|---------|----------------|
| **SoftNCutsLoss** | encoder output (segmentation) | Encourages class-coherent regions | $\alpha = 0.5$ |
| **MSELoss** (or BCE) | decoder output (reconstruction) | Forces faithful reconstruction | $\beta = 0.5$ |

### SoftNCutsLoss

For each class k:
1. Compute class mean brightness: $\mu_k = \frac{E[I \cdot P_k]}{E[P_k] + \epsilon}$
2. Intensity difference: $\Delta = (I - \mu_k)^2$
3. Brightness weights: $W = \exp(-\Delta^2 / \sigma_I^2)$
4. Spatial Gaussian kernel G of size $(2r+1)^3$
5. numerator = $\sum P_k \cdot (P_k \cdot W) * G$
6. denominator = $\sum P_k \cdot W * G$
7. $\mathcal{L}_{\text{NCuts}} = K - \sum_k |num / den|$

**Reference**: `papers_code/CellSeg3D/.../wnet/soft_Ncuts.py`

In [ ]:
import math
import numpy as np
import torch.nn.functional as F
from scipy.stats import norm


class SoftNCutsLoss(nn.Module):
    """3D Soft Normalized Cuts loss.

    Encourages the encoder to produce segmentation maps where each class
    covers a spatially and intensity-coherent region of the volume.

    Uses a convolution-based approximation (Gaussian kernel of radius r)
    instead of computing the full pairwise weight matrix, which would be
    far too expensive for 3D volumes.

    Reference: CellSeg3D soft_Ncuts.py

    Args:
        data_shape:      (H, W, D) spatial shape of input volumes.
        intensity_sigma: scale for brightness affinity kernel (default 1.0).
        spatial_sigma:   scale for spatial Gaussian kernel (default 4.0).
        radius:          radius of the spatial kernel. If None, auto-computed.
    """

    def __init__(self, data_shape, intensity_sigma: float = 1.0,
                 spatial_sigma: float = 4.0, radius: int = 2):
        super().__init__()
        self.intensity_sigma = intensity_sigma
        self.spatial_sigma = spatial_sigma
        self.radius = radius
        self.H, self.W, self.D = data_shape

        # Auto-compute radius if not provided
        if self.radius is None:
            self.radius = min(
                max(5, math.ceil(min(self.H, self.W, self.D) / 20)),
                self.H, self.W, self.D,
            )

        # Pre-compute spatial Gaussian kernel and register as buffer
        kernel = self._build_gaussian_kernel(self.radius, self.spatial_sigma)
        self.register_buffer("kernel", kernel)

    def _build_gaussian_kernel(self, radius: int, sigma: float) -> torch.Tensor:
        """Build a 3D Gaussian kernel of size (2*radius+1)^3."""
        x_2 = np.linspace(-radius, radius, 2 * radius + 1) ** 2
        dist = (
            np.sqrt(
                x_2.reshape(-1, 1, 1)
                + x_2.reshape(1, -1, 1)
                + x_2.reshape(1, 1, -1)
            ) / sigma
        )
        kernel = norm.pdf(dist) / norm.pdf(0)
        kernel = torch.from_numpy(kernel.astype(np.float32))
        return kernel.view(1, 1, *kernel.shape)

    def forward(self, labels: torch.Tensor, inputs: torch.Tensor) -> torch.Tensor:
        """Compute Soft NCuts loss.

        Args:
            labels: predicted class probabilities [B, K, H, W, D].
            inputs: input images [B, C, H, W, D].

        Returns:
            Scalar loss value.
        """
        K = labels.shape[1]  # number of classes
        loss = torch.tensor(0.0, device=labels.device, dtype=labels.dtype)
        kernel = self.kernel

        for k in range(K):
            # Per-class probability map: [B, 1, H, W, D]
            class_probs = labels[:, k].unsqueeze(1)

            # Class mean brightness: E[I * P_k] / E[P_k]
            class_mean = torch.mean(
                inputs * class_probs, dim=(2, 3, 4), keepdim=True
            ) / torch.add(
                torch.mean(class_probs, dim=(2, 3, 4), keepdim=True), 1e-5
            )

            # Squared intensity difference, summed over channels
            diff = (inputs - class_mean).pow(2).sum(dim=1).unsqueeze(1)

            # Brightness weights: exp(-diff^2 / sigma_I^2)
            weights = torch.exp(diff.pow(2).mul(-1.0 / self.intensity_sigma ** 2))

            # Numerator: sum(P_k * conv3d(P_k * W, kernel))
            numerator = torch.sum(
                class_probs
                * F.conv3d(class_probs * weights, kernel, padding=self.radius),
                dim=(1, 2, 3, 4),
            )

            # Denominator: sum(P_k * conv3d(W, kernel))
            denominator = torch.sum(
                class_probs * F.conv3d(weights, kernel, padding=self.radius),
                dim=(1, 2, 3, 4),
            )

            # L1 loss of (num/den) against zero
            loss += nn.L1Loss()(
                numerator / torch.add(denominator, 1e-6),
                torch.zeros_like(numerator),
            )

        return K - loss


print("SoftNCutsLoss defined.")

### Combined training loss

During training, the two losses are combined as:



Where:
- `enc` = encoder output (K-class soft segmentation)
- `dec` = decoder output (reconstructed image)
- `image` = input image
- `alpha` = NCuts weight (default 0.5)
- `beta` = reconstruction weight (default 0.5)

In [ ]:
def get_reconstruction_loss(loss_type: str = "mse") -> nn.Module:
    """Return the reconstruction loss by name.

    Supported: "mse" (MSELoss), "bce" (BCELoss).
    CellSeg3D default is MSE.
    """
    if loss_type == "mse":
        return nn.MSELoss()
    elif loss_type == "bce":
        return nn.BCELoss()
    else:
        raise ValueError(f"Unknown reconstruction loss: {loss_type}. Choose "mse" or "bce".")


print("get_reconstruction_loss defined.")

### Loss smoke test

In [ ]:
def test_losses():
    """Test both losses with a random WNet3D forward pass."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    spatial = (32, 32, 32)

    model = WNet3D().to(device)
    model.eval()

    x = torch.randn(1, 1, *spatial, device=device)

    with torch.no_grad():
        enc, dec = model(x)

    # SoftNCutsLoss
    ncuts_fn = SoftNCutsLoss(
        data_shape=spatial, intensity_sigma=1.0, spatial_sigma=4.0, radius=2
    ).to(device)
    ncuts_val = ncuts_fn(enc, x)
    print(f"SoftNCuts loss: {ncuts_val.item():.4f}")

    # Reconstruction loss (MSE)
    rec_fn = get_reconstruction_loss("mse")
    rec_val = rec_fn(dec, x)
    print(f"MSE rec loss:   {rec_val.item():.4f}")

    # Combined (CellSeg3D default weights)
    alpha, beta = 0.5, 0.5
    total = alpha * ncuts_val + beta * rec_val
    print(f"Combined loss:  {total.item():.4f}  (alpha={alpha}, beta={beta})")

    del model, x, enc, dec, ncuts_fn, rec_fn
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("OK")


test_losses()